In [19]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
xr.set_options(keep_attrs=True)

# ----------- INPUTS -----------
FILES = [
    ("CHIRPS", r"C:\Drought\SPI\SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    ("IMD",    r"C:\Drought\SPI IMD AND ERA5\SPI_IMD_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    ("ERA5",   r"C:\Drought\SPI IMD AND ERA5\SPI_ERA5_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
]
OUT_ROOT = r"C:\Drought\Outputs"

# ----------- SETTINGS ----------
PRIMARY_K = 3
BASE = slice("2001-01-01","2020-12-31")
THRESH = {"D1": -1.0, "D2": -1.5, "D3": -2.0}
KHARIF_MONTHS = [6,7,8,9]
RABI_MONTHS   = [10,11,12,1,2,3]
DROUGHT_YEARS = [2002, 2009, 2015, 2016, 2019]
CHUNKS = {"time": -1, "lat": 200, "lon": 200}

# ---------- HELPERS ----------
def get_spi_var(ds, k=PRIMARY_K):
    v = f"SPI{k}"
    if v in ds.data_vars: return v
    for name in ds.data_vars:
        if name.upper() == v: return name
    raise KeyError(f"{v} not in {list(ds.data_vars)}")

def open_spi(path):
    ds = xr.open_dataset(path, chunks=CHUNKS)
    # orient vars to (time, lat, lon), sort coords ascending
    for v in ds.data_vars:
        ds[v] = ds[v].transpose("time","lat","lon")
    if ds.lat.size>1 and float(ds.lat[1]-ds.lat[0])<0: ds = ds.sortby("lat")
    if ds.lon.size>1 and float(ds.lon[1]-ds.lon[0])<0: ds = ds.sortby("lon")
    if "month" not in ds.coords:
        ds = ds.assign_coords(month=("time", ds["time"].dt.month))
    return ds

def area_weights(lat):
    w = np.cos(np.deg2rad(lat))
    return xr.DataArray(w / w.sum(), dims=["lat"], coords={"lat": lat})

def percent_area(mask_bool, wlat, tdim="time"):
    """
    Weighted % area of True across the grid along a time-like dimension tdim ('time' or 'year').
    mask_bool: DataArray with dims (tdim, lat, lon)
    """
    if tdim not in mask_bool.dims:
        for candidate in ("time","year"):
            if candidate in mask_bool.dims:
                tdim = candidate; break
    ref = mask_bool.isel({tdim: 0})
    w2d = wlat.broadcast_like(ref)
    num = (mask_bool * w2d).sum(("lat","lon"), skipna=True)
    den = (w2d * xr.ones_like(ref)).sum(("lat","lon"))
    return (num / den) * 100.0

def seasonal_mean(da, months):
    return da.where(da["time"].dt.month.isin(months), drop=True)\
             .groupby("time.year").mean("time")  # (year, lat, lon)

def polyfit_trend(y):
    """
    Pixel-wise linear trend of annual drought frequency.
    y: DataArray (year, lat, lon) of fractions.
    """
    # ensure 'year' is a single chunk (tiny), then allow rechunking in gufunc
    y = y.chunk({"year": -1})
    x = np.arange(y.sizes["year"], dtype="float32")
    return xr.apply_ufunc(
        lambda yy: np.polyfit(x, yy, 1)[0],  # slope
        y,
        input_core_dims=[["year"]],
        output_core_dims=[[]],               # drop 'year'
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float32],
        dask_gufunc_kwargs={"allow_rechunk": True},
    )

# ---- plotting ----
def nice_ax(ax):
    ax.grid(True, alpha=0.2, lw=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_ts_pct(ax, ts_df, title):
    ts_df.plot(ax=ax, lw=1.6)
    ax.set_ylabel("% area"); ax.set_title(title, loc="left", fontsize=11)
    nice_ax(ax)

def plot_map(ax, da, title, vmin=None, vmax=None, cmap="RdBu_r"):
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, shading="auto",
                       cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# ---------- PIPELINE ----------
def process_one(name, path):
    print(f"\n=== {name} ===")
    out_dir = Path(OUT_ROOT) / f"SPI_{name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ds  = open_spi(path)
    v   = get_spi_var(ds, PRIMARY_K)
    spi = ds[v]  # (time, lat, lon)

    # baseline QA
    base = spi.sel(time=BASE)
    print(f"Baseline QA {v}: mean≈{float(base.mean('time').mean(('lat','lon')).values):.2f}, "
          f"std≈{float(base.std('time').mean(('lat','lon')).values):.2f}")

    wlat = area_weights(spi["lat"])

    # --- national % area (monthly) ---
    masks = {k: (spi <= thr) for k, thr in THRESH.items()}
    pct_ds = xr.Dataset({k: percent_area(m, wlat, tdim="time") for k, m in masks.items()})
    pct_df = pct_ds.to_dataframe()
    csv_nat = out_dir / "national_pctarea_SPI3_D1_D2_D3_monthly.csv"
    pct_df.to_csv(csv_nat); print("Saved:", csv_nat)

    # --- seasonal (Kharif/Rabi) % area by year ---
    S_kharif = seasonal_mean(spi, KHARIF_MONTHS)  # (year, lat, lon)
    S_rabi   = seasonal_mean(spi, RABI_MONTHS)    # (year, lat, lon)
    kh_pct = percent_area((S_kharif <= -1.0), wlat, tdim="year")
    rb_pct = percent_area((S_rabi   <= -1.0), wlat, tdim="year")
    (out_dir / "seasonal_kharif_pctarea_SPI3.csv").write_text(
        kh_pct.to_dataframe("pct_area").to_csv(index=True)
    )
    (out_dir / "seasonal_rabi_pctarea_SPI3.csv").write_text(
        rb_pct.to_dataframe("pct_area").to_csv(index=True)
    )
    print("Saved:", out_dir / "seasonal_kharif_pctarea_SPI3.csv")
    print("Saved:", out_dir / "seasonal_rabi_pctarea_SPI3.csv")

    # --- maps (NetCDF) ---
    freq_year = (spi <= -1.0).groupby("time.year").mean("time").chunk({"year": -1})
    trend = polyfit_trend(freq_year).rename("freq_trend_per_year")
    kh_map = S_kharif.mean("year").rename("kharif_mean_spi3")
    rb_map = S_rabi.mean("year").rename("rabi_mean_spi3")
    freq_clim = freq_year.mean("year").rename("drought_frequency_fraction")
    maps_ds = xr.Dataset({
        "kharif_mean_spi3": kh_map.astype("float32"),
        "rabi_mean_spi3": rb_map.astype("float32"),
        "drought_frequency_fraction": freq_clim.astype("float32"),
        "freq_trend_per_year": trend.astype("float32"),
    })
    maps_nc = out_dir / "spi3_maps.nc"
    enc = {v: {"zlib": True, "complevel": 4, "_FillValue": np.float32(np.nan)} for v in maps_ds.data_vars}
    maps_ds.to_netcdf(maps_nc, encoding=enc); print("Saved:", maps_nc)

    # --- drought-year outputs ---
    drought_layers = {}
    for yr in DROUGHT_YEARS:
        # national monthly % area subset
        ts_year = pct_df.loc[str(yr)]
        (out_dir / f"national_pctarea_SPI3_D1_D2_D3_{yr}.csv").write_text(
            ts_year.to_csv(index=True)
        )
        spi_year = spi.sel(time=slice(f"{yr}-01-01", f"{yr}-12-31"))
        if spi_year.time.size == 0:
            continue
        min_map = spi_year.min("time").rename(f"min_spi3_{yr}")
        drought_layers[f"min_spi3_{yr}"] = min_map.astype("float32")
        # Kharif mean that year if JASO present
        kh_mask = spi_year["time"].dt.month.isin(KHARIF_MONTHS)
        if bool(kh_mask.any()):
            kh_year = spi_year.where(kh_mask, drop=True).mean("time").rename(f"kharif_mean_spi3_{yr}")
            drought_layers[f"kharif_mean_spi3_{yr}"] = kh_year.astype("float32")

    if drought_layers:
        dy_nc = out_dir / "drought_year_maps.nc"
        xr.Dataset(drought_layers).to_netcdf(
            dy_nc, encoding={k: {"zlib": True, "complevel": 4} for k in drought_layers}
        )
        print("Saved:", dy_nc)

    # --- overview figure ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    plot_ts_pct(axes[0,0], pct_df, f"{name} — National % area in drought (SPI3)")
    plot_map(axes[0,1], kh_map, "Kharif mean SPI3", vmin=-2, vmax=2)
    plot_map(axes[1,0], freq_clim, "Drought frequency (SPI3≤−1)", vmin=0, vmax=1, cmap="viridis")
    plot_map(axes[1,1], trend, "Trend of frequency (fraction/yr)", vmin=-0.02, vmax=0.02, cmap="BrBG")
    fig.suptitle(f"SPI3 overview — {name}", fontsize=14)
    fig.savefig(out_dir / "overview_panels.png", dpi=200)
    plt.close(fig)
    print("Saved:", out_dir / "overview_panels.png")

    # --- per-drought-year figures ---
    for yr in DROUGHT_YEARS:
        ts_year = pct_df.loc[str(yr)]
        if ts_year.empty:
            continue
        min_key = f"min_spi3_{yr}"
        kh_key  = f"kharif_mean_spi3_{yr}"
        min_map = drought_layers.get(min_key, None)
        kh_map_y = drought_layers.get(kh_key, None)

        ncols = 3 if kh_map_y is not None else 2
        fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 4), constrained_layout=True)
        if ncols == 2:
            ax0, ax1 = axes
        else:
            ax0, ax1, ax2 = axes

        plot_ts_pct(ax0, ts_year, f"{name} — % area (SPI3) {yr}")
        plot_map(ax1, min_map, f"Min SPI3 in {yr}", vmin=-3, vmax=1)
        if ncols == 3:
            plot_map(ax2, kh_map_y, f"Kharif mean SPI3 ({yr})", vmin=-2, vmax=2)
        fig.savefig(out_dir / f"drought_{yr}_panels.png", dpi=200)
        plt.close(fig)
        print("Saved:", out_dir / f"drought_{yr}_panels.png")

# ----------- RUN ------------
def main():
    Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
    for name, path in FILES:
        if not Path(path).exists():
            print(f"Missing file for {name}: {path} — skipping.")
            continue
        process_one(name, path)
    print("\nAll done.")

if __name__ == "__main__":
    main()



=== CHIRPS ===


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


Baseline QA SPI3: mean≈0.02, std≈0.97
Saved: C:\Drought\Outputs\SPI_CHIRPS\national_pctarea_SPI3_D1_D2_D3_monthly.csv
Saved: C:\Drought\Outputs\SPI_CHIRPS\seasonal_kharif_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_CHIRPS\seasonal_rabi_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_CHIRPS\spi3_maps.nc


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\reductions.py:621: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)


Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_year_maps.nc
Saved: C:\Drought\Outputs\SPI_CHIRPS\overview_panels.png
Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_2002_panels.png
Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_2009_panels.png
Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_2015_panels.png
Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_2016_panels.png
Saved: C:\Drought\Outputs\SPI_CHIRPS\drought_2019_panels.png

=== IMD ===


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


Baseline QA SPI3: mean≈0.01, std≈0.99
Saved: C:\Drought\Outputs\SPI_IMD\national_pctarea_SPI3_D1_D2_D3_monthly.csv
Saved: C:\Drought\Outputs\SPI_IMD\seasonal_kharif_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_IMD\seasonal_rabi_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_IMD\spi3_maps.nc
Saved: C:\Drought\Outputs\SPI_IMD\drought_year_maps.nc
Saved: C:\Drought\Outputs\SPI_IMD\overview_panels.png
Saved: C:\Drought\Outputs\SPI_IMD\drought_2002_panels.png
Saved: C:\Drought\Outputs\SPI_IMD\drought_2009_panels.png
Saved: C:\Drought\Outputs\SPI_IMD\drought_2015_panels.png
Saved: C:\Drought\Outputs\SPI_IMD\drought_2016_panels.png
Saved: C:\Drought\Outputs\SPI_IMD\drought_2019_panels.png

=== ERA5 ===


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


Baseline QA SPI3: mean≈0.06, std≈0.96
Saved: C:\Drought\Outputs\SPI_ERA5\national_pctarea_SPI3_D1_D2_D3_monthly.csv
Saved: C:\Drought\Outputs\SPI_ERA5\seasonal_kharif_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_ERA5\seasonal_rabi_pctarea_SPI3.csv
Saved: C:\Drought\Outputs\SPI_ERA5\spi3_maps.nc
Saved: C:\Drought\Outputs\SPI_ERA5\drought_year_maps.nc
Saved: C:\Drought\Outputs\SPI_ERA5\overview_panels.png
Saved: C:\Drought\Outputs\SPI_ERA5\drought_2002_panels.png
Saved: C:\Drought\Outputs\SPI_ERA5\drought_2009_panels.png
Saved: C:\Drought\Outputs\SPI_ERA5\drought_2015_panels.png
Saved: C:\Drought\Outputs\SPI_ERA5\drought_2016_panels.png
Saved: C:\Drought\Outputs\SPI_ERA5\drought_2019_panels.png

All done.
